# Chapter 2 — Multi-Armed Bandits

Learning objectives:
- Implement a bandit environment with Gaussian rewards
- Build epsilon-greedy agent
- Compare epsilon-greedy vs greedy visually

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
print("Ready")

## Exercise 1 — Bandit environment

Implement a 5-armed bandit with Gaussian rewards.

In [ ]:
class BanditEnv:
    """k-armed bandit with Gaussian rewards."""
    def __init__(self, k=5, seed=42):
        rng = np.random.default_rng(seed)
        # TODO: self.means = rng.normal(0, 1, k)  (true means)
        self.means = None
        self.k = k

    def pull(self, arm):
        # TODO: return self.means[arm] + rng.normal()
        # (use np.random.normal or keep a rng)
        pass

env = BanditEnv(k=5)
print("True means:", env.means)
print("Sample from arm 0:", env.pull(0))

In [ ]:
class BanditEnv:
    def __init__(self, k=5, seed=42):
        self.rng = np.random.default_rng(seed)
        self.means = self.rng.normal(0, 1, k)
        self.k = k

    def pull(self, arm):
        return self.means[arm] + self.rng.standard_normal()

env = BanditEnv(k=5)
print("True means:", np.round(env.means, 3))
for _ in range(3):
    print("Arm 0 reward:", round(env.pull(0), 3))

## Exercise 2 — Epsilon-greedy agent

Implement epsilon-greedy: with probability ε choose random arm, else choose arm with highest Q.

In [ ]:
def epsilon_greedy(Q, epsilon, rng=None):
    """Return an action index using epsilon-greedy."""
    if rng is None:
        rng = np.random.default_rng()
    # TODO: with prob epsilon pick random arm, else pick argmax(Q)
    pass

# Test with Q=[0.1, 0.5, 0.3], epsilon=0.0 -> should always return 1
Q_test = np.array([0.1, 0.5, 0.3])
print([epsilon_greedy(Q_test, 0.0) for _ in range(5)])   # [1,1,1,1,1]

In [ ]:
def epsilon_greedy(Q, epsilon, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    if rng.random() < epsilon:
        return int(rng.integers(len(Q)))
    return int(np.argmax(Q))

Q_test = np.array([0.1, 0.5, 0.3])
rng = np.random.default_rng(0)
print([epsilon_greedy(Q_test, 0.0, rng) for _ in range(5)])   # [1,1,1,1,1]

## Exercise 3 — Full training loop (10-armed testbed)

Run epsilon-greedy and greedy for 1000 steps, average over 200 runs.

In [ ]:
def run_bandit(k=10, epsilon=0.1, steps=1000, n_runs=200, seed=0):
    rng = np.random.default_rng(seed)
    avg_rewards = np.zeros(steps)
    for run in range(n_runs):
        env = BanditEnv(k=k, seed=run)
        Q = np.zeros(k)
        N = np.zeros(k, dtype=int)
        for t in range(steps):
            action = epsilon_greedy(Q, epsilon, rng)
            reward = env.pull(action)
            N[action] += 1
            Q[action] += (reward - Q[action]) / N[action]   # incremental mean
            avg_rewards[t] += reward
    return avg_rewards / n_runs

rewards_eg = run_bandit(epsilon=0.1)
rewards_greedy = run_bandit(epsilon=0.0)

plt.figure(figsize=(8, 3))
plt.plot(rewards_eg, label='ε=0.1 (epsilon-greedy)')
plt.plot(rewards_greedy, label='ε=0 (greedy)')
plt.xlabel('Steps')
plt.ylabel('Average reward')
plt.title('Epsilon-greedy vs Greedy (10-armed testbed)')
plt.legend()
plt.tight_layout()
plt.show()